# 1. Prepare the data (CPU notebook)

The training notebook reads a **cache** (`.npy` files, ~4 MB per patch) from a Kaggle dataset.
There are two ways to make it:

**A. Build it on another machine and upload it (recommended).** On a Linux machine with the
data extracted (for example the RTX A4000 box):
```
python scripts/prepare_sen12mscr.py cache --config configs/full.yaml --set data.root=/data/sen12ms_cr data.cache=/data/cache
pip install kaggle   # API token in ~/.kaggle/kaggle.json (Kaggle > Settings > API)
python kaggle/kaggle_tools.py publish --dir /data/cache --slug diewolke-cache
```
Then skip this notebook.

**B. Build it here.** This notebook downloads the archives from TUM to `/kaggle/temp`, then
extracts S1 first, uses its file names to pick the train/val/test
patches, and keeps only those patches from the two large S2 archives. Then it builds the cache in
`/kaggle/working/cache`. Save the version, and its output becomes the input of the training
notebook. Peak disk is about the extracted S1 data plus the chosen patches; the tool checks free
space before the S2 archives and stops early if they won't fit.

- Settings: Accelerator **None**, Internet **On** (needs a phone-verified account).
- The archives come from TUM over **rsync**, all three in parallel. TUM's HTTPS share is
  throttled to ~0.1 MB/s; rsync gave ~8.7 MB/s total from Kaggle (three connections), so the
  70 GB of spring archives take about 3.5 h (the 35 GB S2 archive is the slowest). The archives
  go to `/kaggle/temp` (over 1 TB free there; not saved). Dropped connections resume, and every
  archive is checked against TUM's `checksums.sha512`.
- Put the rsync password from https://mediatum.ub.tum.de/1554803 in `RSYNC_PASSWORD` below
  (or in a Kaggle secret of that name). Don't commit it.
- Everything must finish in one 12 h session: download ~3.5 h, then extraction and caching.
- `/kaggle/working` is saved with the version, and users report a ~20 GB limit for it. So the
  default below caches 2,500 training and 1,500 test patches (about 17 GB). If you need the full
  test split, run this notebook a second time with `--splits test` only, and attach both outputs.

**A larger training set (shards).** The full spring training split is far bigger than 20 GB, so it is
cached in shards of 4,500 patches (~18 GB each). Set `SHARD = 0` below and save a version; then make a
copy of this notebook per further shard (`SHARD = 1`, `2`, ...) so their outputs stay separate (the
copies can run at the same time). The first shard's log prints how many shards there are. Each shard
holds training patches only; val and test still come from the original cache. In the training
notebook, attach every shard **and** the original cache: `find-cache` merges the shards on local disk.

In [ ]:
import os, glob, shutil, time
T0 = time.time()                      # session start, used to stop training before Kaggle's 12 h limit
REPO_URL = 'https://github.com/Rasvanth28/DieWolke.git'  # needs Internet on; '' = use the code dataset
REPO_BRANCH = 'shrey'                 # the branch with the NDVI fix
CODE = '/kaggle/working/DieWolke'
if not os.path.exists(CODE):
    if REPO_URL:
        !git clone -q --depth 1 -b {REPO_BRANCH} {REPO_URL} {CODE}
        if os.path.exists(CODE): print('code cloned from', REPO_URL, REPO_BRANCH)
    if not os.path.exists(CODE):  # no URL or no Internet: the code uploaded as a Kaggle dataset (DieWolke folder, no data or runs)
        hits = sorted(glob.glob('/kaggle/input/**/diewolke/__init__.py', recursive=True), key=len)
        assert hits, 'Add the DieWolke code as an input dataset (Add Input > Your Datasets), or set REPO_URL'
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE,
                        ignore=shutil.ignore_patterns('*.npy', 'runs', '.git', '.venv', '__pycache__'))
%cd {CODE}
!python kaggle/kaggle_tools.py env

In [ ]:
import os
!pip install -q rasterio s2cloudless
!which rsync > /dev/null || apt-get -qq install -y rsync > /dev/null
RSYNC_PASSWORD = ''   # from https://mediatum.ub.tum.de/1554803; or leave empty and use a Kaggle secret
if not RSYNC_PASSWORD:
    from kaggle_secrets import UserSecretsClient
    RSYNC_PASSWORD = UserSecretsClient().get_secret('RSYNC_PASSWORD')
os.environ['RSYNC_PASSWORD'] = RSYNC_PASSWORD
RSYNC = 'rsync://m1554803@dataserv.ub.tum.de/m1554803'

Archive sizes on the server (also checks that rsync and the password work):

In [ ]:
!rsync --list-only {RSYNC}/ | grep spring

In [ ]:
SHARD = None   # None: the original cache (2,500 train + val + test). 0, 1, 2, ...: one training shard of 4,500 patches
if SHARD is None:
    !python kaggle/kaggle_tools.py prepare --rsync {RSYNC} --out /kaggle/working/cache --n_train 2500 --n_test 1500 --delete_tifs
else:
    !python kaggle/kaggle_tools.py prepare --rsync {RSYNC} --out /kaggle/working/cache --shard {SHARD} --delete_tifs
!ls -la /kaggle/working/cache | head -30

Now **Save Version > Save & Run All** (or Quick Save if it already ran). In the training
notebook, use **Add Input > Notebook Output** and pick this notebook. The tools find the cache
automatically.